In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [5]:
import os
import re
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

# =========================
# Configuration
# =========================
CSV_PATH    = "/kaggle/input/datasets/malakaboelmagd/xai-data/final_preprocessed_data.csv"
IMAGES_DIR  = "/kaggle/input/datasets/ahmedmohsen2005/xai-preprocessed-skin-lesion-17k-dataset/final_processed_images"
OUTPUT_DIR  = "splitted-data"
RANDOM_SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15
LABEL_COLUMN = "class"
IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]

# =========================
# Normalize image name
# (same logic as Hybrid model — extracts 7-digit ISIC id)
# =========================
def normalize_name(fname):
    """
    CSV contains names like:
      ISIC2019_0069065_mel.jpg  →  ISIC_0069065.jpg
      ISIC_0022558_oth.jpg      →  ISIC_0022558.jpg
      ISIC2020_6363862_oth.jpg  →  ISIC_6363862.jpg
    Extracts the 7-digit number and builds ISIC_XXXXXXX.jpg
    """
    x    = str(fname).replace(".jpg", "").replace(".jpeg", "")
    nums = re.findall(r'(\d{7})', x)
    if nums:
        return f"ISIC_{nums[-1]}.jpg"
    return None

# =========================
# Find image on disk
# =========================
def find_image_file(normalized_name):
    """Try to find the normalized image name in IMAGES_DIR."""
    if normalized_name is None:
        return None
    path = os.path.join(IMAGES_DIR, normalized_name)
    if os.path.exists(path):
        return path
    # fallback: try other extensions
    base = normalized_name.rsplit(".", 1)[0]
    for ext in IMAGE_EXTENSIONS:
        p = os.path.join(IMAGES_DIR, base + ext)
        if os.path.exists(p):
            return p
    return None

# =========================
# Copy split to disk
# =========================
def copy_split(split_df, split_name):
    split_folder  = os.path.join(OUTPUT_DIR, split_name)
    images_folder = os.path.join(split_folder, "images")
    os.makedirs(images_folder, exist_ok=True)

    split_df.to_csv(os.path.join(split_folder, f"{split_name}.csv"), index=False)

    missing = []
    for _, row in split_df.iterrows():
        normalized = row["image_fixed"]
        src = find_image_file(normalized)
        if src is None:
            missing.append(row["image"])
            continue
        dst = os.path.join(images_folder, os.path.basename(src))
        shutil.copy2(src, dst)

    if missing:
        print(f"[WARNING] {len(missing)} images still not found in {split_name}:")
        print(missing[:5], "..." if len(missing) > 5 else "")
    else:
        print(f"[✓] {split_name}: all images copied successfully")

    print(f"[DONE] {split_name}: {len(split_df)} samples")

# =========================
# Load CSV & fix names
# =========================
df = pd.read_csv(CSV_PATH)
print(f"Total rows in CSV: {len(df)}")

# Apply normalize_name — same as Hybrid/GNN models
df["image_fixed"] = df["image"].apply(normalize_name)

# Drop rows where name couldn't be parsed
before = len(df)
df = df[df["image_fixed"].notna()].reset_index(drop=True)
print(f"Rows after name fix: {len(df)}  (dropped {before - len(df)} unparseable)")

# Quick check — how many images actually exist on disk?
df["_exists"] = df["image_fixed"].apply(lambda x: find_image_file(x) is not None)
found   = df["_exists"].sum()
missing = (~df["_exists"]).sum()
print(f"\nImages found on disk : {found}")
print(f"Images NOT found     : {missing}")

if missing > 0:
    print("\nSample missing (original names):")
    print(df[~df["_exists"]]["image"].head(5).tolist())
    print("\nSample missing (normalized names):")
    print(df[~df["_exists"]]["image_fixed"].head(5).tolist())

    print("\nSample files actually in IMAGES_DIR:")
    sample_files = os.listdir(IMAGES_DIR)[:10]
    print(sample_files)

# Keep only rows where image exists
df = df[df["_exists"]].drop(columns=["_exists"]).reset_index(drop=True)
print(f"\nFinal usable samples: {len(df)}")
print(f"Class distribution:\n{df[LABEL_COLUMN].value_counts()}")

# =========================
# Train / Val / Test Split
# =========================
stratify_col = df[LABEL_COLUMN] if LABEL_COLUMN in df.columns else None

train_df, temp_df = train_test_split(
    df,
    test_size     = 1 - TRAIN_RATIO,
    random_state  = RANDOM_SEED,
    stratify      = stratify_col,
    shuffle       = True
)

temp_stratify = temp_df[LABEL_COLUMN] if LABEL_COLUMN in temp_df.columns else None

val_df, test_df = train_test_split(
    temp_df,
    test_size    = TEST_RATIO / (VAL_RATIO + TEST_RATIO),
    random_state = RANDOM_SEED,
    stratify     = temp_stratify,
    shuffle      = True
)

print(f"\nSplit sizes:")
print(f"  Train : {len(train_df)}")
print(f"  Val   : {len(val_df)}")
print(f"  Test  : {len(test_df)}")

# =========================
# Copy to disk
# =========================
os.makedirs(OUTPUT_DIR, exist_ok=True)
copy_split(train_df, "train")
copy_split(val_df,   "val")
copy_split(test_df,  "test")

# =========================
# Summary
# =========================
print("\n=== Split Summary ===")
print(f"Train : {len(train_df)} samples")
print(f"Val   : {len(val_df)} samples")
print(f"Test  : {len(test_df)} samples")
print(f"Output: {OUTPUT_DIR}/")
print("""
Structure:
  dataset_split/
  ├── train/
  │   ├── train.csv
  │   └── images/
  ├── val/
  │   ├── val.csv
  │   └── images/
  └── test/
      ├── test.csv
      └── images/
""")

Total rows in CSV: 17059
Rows after name fix: 17059  (dropped 0 unparseable)

Images found on disk : 17059
Images NOT found     : 0

Final usable samples: 17059
Class distribution:
class
1    8530
0    8529
Name: count, dtype: int64

Split sizes:
  Train : 11941
  Val   : 2559
  Test  : 2559
[✓] train: all images copied successfully
[DONE] train: 11941 samples
[✓] val: all images copied successfully
[DONE] val: 2559 samples
[✓] test: all images copied successfully
[DONE] test: 2559 samples

=== Split Summary ===
Train : 11941 samples
Val   : 2559 samples
Test  : 2559 samples
Output: splitted-data/

Structure:
  dataset_split/
  ├── train/
  │   ├── train.csv
  │   └── images/
  ├── val/
  │   ├── val.csv
  │   └── images/
  └── test/
      ├── test.csv
      └── images/



In [6]:
import os
import zipfile
from IPython.display import FileLink, display

# =========================
# Configuration
# =========================
OUTPUT_DIR = "splitted-data"   # same as split script

# =========================
# Zip each split separately
# =========================
splits = ["train", "val", "test"]

for split in splits:
    split_folder = os.path.join(OUTPUT_DIR, split)
    zip_path     = f"{split}.zip"

    if not os.path.exists(split_folder):
        print(f"[SKIP] {split_folder} not found — run the split script first")
        continue

    print(f"Zipping {split}...", end=" ")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(split_folder):
            for file in files:
                full_path = os.path.join(root, file)
                arcname   = os.path.relpath(full_path, OUTPUT_DIR)
                zf.write(full_path, arcname)

    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"done  ({size_mb:.1f} MB)")

# =========================
# Download links
# =========================
print("\n=== Download Links ===")
for split in splits:
    zip_path = f"{split}.zip"
    if os.path.exists(zip_path):
        display(FileLink(zip_path, result_html_prefix=f"{split}: "))

Zipping train... done  (198.3 MB)
Zipping val... done  (42.5 MB)
Zipping test... done  (42.6 MB)

=== Download Links ===


/kaggle/working/train.zip

/kaggle/working/val.zip

/kaggle/working/test.zip